[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pflahert/hd-stats-workshop/blob/main/notebooks/lab2.ipynb)

# Lab 2: The Histogram That Was Too Wide
## FDR in Practice

**Day 1, 1:00–2:00** | Learning Outcomes: LO1, LO3

In this lab you will:
1. Implement the BH step-up procedure by hand and verify it
2. Estimate $\pi_0$ using Storey's method
3. Compute q-values manually
4. Explore the empirical null through simulation
5. Create a volcano plot
6. Compare multiple FDR methods

We continue with the ALL dataset. You already have 12,625 p-values from Lab 1. Now you'll learn what to do with them properly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

np.random.seed(2026)

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('default')

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

In [ ]:
# Load ALL dataset
# Try GitHub URL first; if the repo is private, upload the CSVs manually
url_base = "https://raw.githubusercontent.com/pflahert/hd-stats-workshop/main/data/"
try:
    expr = pd.read_csv(url_base + "all_expression.csv", index_col=0)
    meta = pd.read_csv(url_base + "all_metadata.csv", dtype={'sample_id': str})
    print("Loaded data from GitHub.")
except Exception:
    print("Could not fetch from GitHub (repo may be private).")
    print("Upload all_expression.csv and all_metadata.csv when prompted.")
    from google.colab import files
    uploaded = files.upload()
    expr = pd.read_csv("all_expression.csv", index_col=0)
    meta = pd.read_csv("all_metadata.csv", dtype={'sample_id': str})
    print("Loaded data from uploaded files.")

# Ensure column names are strings for safe indexing
expr.columns = expr.columns.astype(str)

# Verify alignment
assert set(meta['sample_id']) == set(expr.columns), \
    "Sample IDs in metadata do not match expression matrix columns!"

print(f"Expression matrix: {expr.shape[0]} genes x {expr.shape[1]} samples")
print(f"Groups: {meta['subtype'].value_counts().to_dict()}")
print(f"Ratio p/n: {expr.shape[0] / expr.shape[1]:.1f}")

# Split samples by subtype
b_samples = meta.loc[meta['subtype'] == 'B', 'sample_id'].values
t_samples = meta.loc[meta['subtype'] == 'T', 'sample_id'].values

# Compute two-sample t-tests for all genes (Welch's t-test, consistent with Lab 1)
pvals = np.zeros(expr.shape[0])
log2fc = np.zeros(expr.shape[0])  # log2 fold change (mean difference of log2 values)
gene_names = expr.index.values

for i, gene in enumerate(gene_names):
    b_expr = expr.loc[gene, b_samples].values.astype(float)
    t_expr = expr.loc[gene, t_samples].values.astype(float)
    stat, p = stats.ttest_ind(b_expr, t_expr, equal_var=False)
    pvals[i] = p
    log2fc[i] = np.mean(b_expr) - np.mean(t_expr)  # on log2 scale, difference = log2 FC

print(f"\nComputed p-values for {len(pvals)} genes")
print(f"Significant at alpha = 0.05 (uncorrected): {np.sum(pvals < 0.05)}")

# p-value histogram
fig, ax = plt.subplots()
ax.hist(pvals, bins=100, density=True, color='steelblue', edgecolor='white', alpha=0.8)
ax.axhline(y=1, color='red', linestyle='--', linewidth=2, label='Uniform(0,1)')
ax.set_xlabel('p-value')
ax.set_ylabel('Density')
ax.set_title('p-value histogram: B-cell vs T-cell ALL (Welch t-tests)')
ax.legend()
plt.tight_layout()
plt.show()

---

## Part 1: BH Step-Up Procedure (By Hand)

You're going to implement the Benjamini-Hochberg procedure yourself. The idea is beautifully geometric:

1. Sort the p-values from smallest to largest: $p_{(1)} \le p_{(2)} \le \cdots \le p_{(m)}$
2. Compare each sorted p-value to its BH threshold line: $\frac{i}{m} \cdot \alpha$
3. Find the largest $k$ where $p_{(k)} \le \frac{k}{m} \cdot \alpha$
4. Reject hypotheses $1, 2, \ldots, k$

This is the step-up procedure from Lecture 2. Implement it below and verify it matches `statsmodels`.

In [ ]:
alpha = 0.05
m = len(pvals)

# Step 1: Sort p-values and keep track of original indices
sorted_indices = np.argsort(pvals)
sorted_pvals = pvals[sorted_indices]

# Step 2: Compute BH threshold for each rank
ranks = np.arange(1, m + 1)
bh_threshold = (ranks / m) * alpha

# Step 3: Find the largest k where p_(k) <= (k/m) * alpha
below_line = sorted_pvals <= bh_threshold
if np.any(below_line):
    k_bh = np.max(np.where(below_line)[0]) + 1  # +1 because 0-indexed
else:
    k_bh = 0

print(f"BH procedure (hand-coded): {k_bh} discoveries at FDR <= {alpha}")

# Verify with statsmodels
reject_sm, padj_sm, _, _ = multipletests(pvals, alpha=alpha, method='fdr_bh')
print(f"BH procedure (statsmodels): {np.sum(reject_sm)} discoveries at FDR <= {alpha}")
print(f"Results match: {k_bh == np.sum(reject_sm)}")

# Plot: sorted p-values vs BH line
fig, ax = plt.subplots(figsize=(10, 6))

# Only plot the first 2000 for clarity
n_plot = min(2000, m)
ax.scatter(ranks[:n_plot], sorted_pvals[:n_plot], s=1, alpha=0.5, color='steelblue', label='Sorted p-values')
ax.plot(ranks[:n_plot], bh_threshold[:n_plot], color='red', linewidth=2, label=f'BH line (slope = {alpha}/m)')
ax.axvline(x=k_bh, color='green', linestyle='--', linewidth=1.5, label=f'k* = {k_bh} (cutoff)')

ax.set_xlabel('Rank (i)')
ax.set_ylabel('p-value')
ax.set_title('BH Step-Up Procedure: Sorted p-values vs. Threshold Line')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

**Exercise 1.1**: Your hand-coded BH and `statsmodels` should agree. How many discoveries at FDR $\le 0.05$?

**YOUR ANSWER:**

---

## Part 2: Storey's $\pi_0$ Estimator (By Hand)

The BH procedure implicitly assumes all hypotheses could be null ($\pi_0 = 1$). But look at your p-value histogram: many genes clearly have signal. If we estimate the fraction of true nulls $\pi_0$, we can be more powerful.

**Key insight**: For any threshold $\lambda$, p-values above $\lambda$ are mostly from true nulls (since null p-values are uniform). So:

$$\hat{\pi}_0(\lambda) = \frac{\#\{p_i > \lambda\}}{m(1 - \lambda)}$$

We compute this for a range of $\lambda$ values and pick a stable estimate.

In [ ]:
def estimate_pi0(pvals, lambda_range=np.arange(0.05, 0.96, 0.05)):
    """
    Estimate pi0 using Storey's method.
    
    For each lambda, estimate pi0 as the fraction of p-values above lambda,
    divided by (1 - lambda). Then select a stable estimate.
    
    Note: Storey (2002) uses a spline smoother to find the stable plateau.
    Here we use a simpler approach: the median of the last few estimates.
    The highest-lambda estimate alone can be noisy because few p-values
    remain in the numerator.
    
    Parameters
    ----------
    pvals : array-like
        The p-values from the hypothesis tests.
    lambda_range : array-like
        The range of lambda values to try.
    
    Returns
    -------
    pi0_hat : float
        The estimated proportion of true nulls.
    pi0_estimates : array
        The pi0 estimates at each lambda.
    lambda_range : array
        The lambda values used.
    """
    m = len(pvals)
    pi0_estimates = np.array([
        np.sum(pvals > lam) / (m * (1 - lam))
        for lam in lambda_range
    ])
    
    # Use the median of the last 3 estimates for stability
    # (Storey's R package qvalue uses a spline; this is a simpler approximation)
    pi0_hat = min(float(np.median(pi0_estimates[-3:])), 1.0)
    
    return pi0_hat, pi0_estimates, lambda_range


pi0_hat, pi0_estimates, lambda_vals = estimate_pi0(pvals)

print(f"Estimated pi0: {pi0_hat:.4f}")
print(f"Interpretation: ~{pi0_hat*100:.1f}% of genes are estimated to be truly null")
print(f"              ~{(1-pi0_hat)*100:.1f}% of genes show a real difference between B and T cells")

# Plot pi0 estimates vs lambda
fig, ax = plt.subplots()
ax.plot(lambda_vals, pi0_estimates, 'o-', color='steelblue', markersize=5)
ax.axhline(y=pi0_hat, color='red', linestyle='--', linewidth=1.5, 
           label=f'$\\hat{{\\pi}}_0$ = {pi0_hat:.3f}')
ax.set_xlabel('$\\lambda$')
ax.set_ylabel('$\\hat{\\pi}_0(\\lambda)$')
ax.set_title("Storey's $\\pi_0$ Estimate vs. $\\lambda$")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

**Exercise 2.1**: What is your estimate of $\pi_0$? What does it mean biologically -- roughly what fraction of genes show no difference between B-cell and T-cell ALL?

**YOUR ANSWER:**

---

## Part 3: Q-Values (Manual Computation)

The **q-value** of a gene is the minimum FDR at which that gene would be called significant. It incorporates the $\pi_0$ estimate, making it more powerful than BH when many genes are truly null.

The q-value for the gene with the $i$-th smallest p-value is:

$$q_{(i)} = \min_{j \ge i} \frac{\hat{\pi}_0 \cdot m \cdot p_{(j)}}{j}$$

We compute this from the bottom up, taking cumulative minima.

In [ ]:
def compute_qvalues(pvals, pi0):
    """
    Compute q-values given p-values and an estimate of pi0.
    
    Parameters
    ----------
    pvals : array-like
        Raw p-values.
    pi0 : float
        Estimated proportion of true nulls.
    
    Returns
    -------
    qvals : array
        Q-values in the same order as the input p-values.
    """
    m = len(pvals)
    sorted_indices = np.argsort(pvals)
    sorted_pvals = pvals[sorted_indices]
    
    # Compute raw q-values: pi0 * m * p_(i) / i
    ranks = np.arange(1, m + 1)
    raw_qvals = pi0 * m * sorted_pvals / ranks
    
    # Enforce monotonicity: q_(i) <= q_(i+1) by taking cumulative min from the right
    qvals_sorted = np.minimum.accumulate(raw_qvals[::-1])[::-1]
    qvals_sorted = np.minimum(qvals_sorted, 1.0)  # cap at 1
    
    # Map back to original order
    qvals = np.empty(m)
    qvals[sorted_indices] = qvals_sorted
    
    return qvals


qvals = compute_qvalues(pvals, pi0_hat)

# Compare discovery counts at different thresholds
thresholds = [0.01, 0.05, 0.10]
print("FDR Threshold | BH Discoveries | Q-value Discoveries")
print("-" * 55)
for t in thresholds:
    _, padj_bh, _, _ = multipletests(pvals, alpha=t, method='fdr_bh')
    n_bh = np.sum(padj_bh <= t)
    n_qval = np.sum(qvals <= t)
    print(f"    {t:.2f}       |     {n_bh:5d}      |       {n_qval:5d}")

**Exercise 3.1**: How do the q-value discovery counts compare to BH? Why might they differ?

(Hint: q-values incorporate the $\pi_0$ estimate. When many genes are truly null, $\pi_0$ is close to 1 and the methods are similar. When many genes have signal, $\pi_0 < 1$ and q-values can be more powerful.)

**YOUR ANSWER:**

---

## Part 4: The Empirical Null (Simulation)

Efron's insight: sometimes the theoretical null $N(0,1)$ is wrong. When you convert p-values to z-scores, the actual null z-scores might be **wider** or **shifted** compared to the standard normal. This can happen due to unobserved confounders, correlations among genes, or other systematic effects.

Let's examine the z-score distribution from our real data.

In [ ]:
np.random.seed(2026)

# Convert p-values to z-scores (two-sided: use the sign of fold change)
z_scores = stats.norm.ppf(1 - pvals / 2) * np.sign(fold_changes)

# Remove any infinite z-scores (from p-values exactly 0 or 1)
finite_mask = np.isfinite(z_scores)
z_finite = z_scores[finite_mask]
print(f"Finite z-scores: {len(z_finite)} out of {len(z_scores)}")

# Plot histogram of z-scores with theoretical null overlay
fig, ax = plt.subplots(figsize=(10, 6))

# Histogram
ax.hist(z_finite, bins=120, density=True, color='steelblue', edgecolor='white', 
        alpha=0.7, label='Observed z-scores')

# Theoretical null: N(0, 1)
z_grid = np.linspace(-6, 6, 300)
ax.plot(z_grid, stats.norm.pdf(z_grid, 0, 1), 'r-', linewidth=2.5, 
        label='Theoretical null N(0, 1)')

# Empirical null: fit a normal to central z-scores (|z| < 2)
central_z = z_finite[np.abs(z_finite) < 2]
mu_emp, sigma_emp = stats.norm.fit(central_z)
ax.plot(z_grid, stats.norm.pdf(z_grid, mu_emp, sigma_emp), 'g--', linewidth=2.5,
        label=f'Empirical null N({mu_emp:.2f}, {sigma_emp:.2f}$^2$)')

ax.set_xlabel('z-score')
ax.set_ylabel('Density')
ax.set_title('Z-score Distribution: Theoretical vs. Empirical Null')
ax.legend(fontsize=11)
ax.set_xlim(-7, 7)
plt.tight_layout()
plt.show()

print(f"\nEmpirical null parameters:")
print(f"  Mean (delta_0):  {mu_emp:.4f}  (theoretical: 0)")
print(f"  SD (sigma_0):    {sigma_emp:.4f}  (theoretical: 1)")
if sigma_emp > 1.05:
    print(f"  --> The empirical null is WIDER than N(0,1). This suggests hidden variability.")
elif sigma_emp < 0.95:
    print(f"  --> The empirical null is NARROWER than N(0,1). Unusual.")
else:
    print(f"  --> The empirical null is close to N(0,1). The theoretical null is reasonable.")

**Exercise 4.1**: If $\sigma_0 > 1$ (the empirical null is wider than the theoretical), what happens if you use the theoretical null $N(0,1)$ instead? Would you get more or fewer discoveries? Would they be trustworthy?

**YOUR ANSWER:**

---

## Part 5: Volcano Plot

A volcano plot shows fold change ($x$-axis) vs. statistical significance ($y$-axis) for every gene. It's the standard visualization for differential expression results.

In [ ]:
# Volcano plot: -log10(p-value) vs. log2 fold change
neg_log10_p = -np.log10(pvals)

# BH significance
reject_bh, padj_bh, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')

fig, ax = plt.subplots(figsize=(10, 7))

# Non-significant genes
ns_mask = ~reject_bh
ax.scatter(log2fc[ns_mask], neg_log10_p[ns_mask], 
           s=3, alpha=0.3, color='grey', label='Not significant')

# Significant genes
sig_mask = reject_bh
ax.scatter(log2fc[sig_mask], neg_log10_p[sig_mask], 
           s=8, alpha=0.6, color='crimson', label=f'BH significant (FDR < 0.05, n={np.sum(sig_mask)})')

# Label top 5 most significant genes
top5_idx = np.argsort(pvals)[:5]
for idx in top5_idx:
    ax.annotate(gene_names[idx], 
                xy=(log2fc[idx], neg_log10_p[idx]),
                xytext=(5, 5), textcoords='offset points',
                fontsize=9, fontweight='bold',
                arrowprops=dict(arrowstyle='-', color='black', lw=0.5))

# Uncorrected threshold
ax.axhline(y=-np.log10(0.05), color='blue', linestyle=':', alpha=0.5, label='p = 0.05 (uncorrected)')

# BH-adjusted threshold: the largest raw p-value among BH-rejected genes
if np.any(reject_bh):
    bh_cutoff_pval = np.max(pvals[reject_bh])
    ax.axhline(y=-np.log10(bh_cutoff_pval), color='green', linestyle='--', alpha=0.7,
               label=f'BH cutoff (p = {bh_cutoff_pval:.4f})')

ax.set_xlabel('Log2 Fold Change (B − T)', fontsize=13)
ax.set_ylabel('$-\\log_{10}$(p-value)', fontsize=13)
ax.set_title('Volcano Plot: B-cell vs. T-cell ALL', fontsize=14)
ax.legend(fontsize=10, loc='upper right')
plt.tight_layout()
plt.show()

---

## Part 6: Method Comparison

Let's bring all the methods together in one summary table.

In [ ]:
# Bonferroni
reject_bonf, _, _, _ = multipletests(pvals, alpha=0.05, method='bonferroni')

# BH
reject_bh, _, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')

# Q-values (using our manual computation)
reject_qval = qvals <= 0.05

# Summary table
summary = pd.DataFrame({
    'Method': ['Uncorrected (alpha = 0.05)', 
               'Bonferroni (FWER <= 0.05)', 
               'BH (FDR <= 0.05)', 
               'Q-value (FDR <= 0.05)'],
    'Discoveries': [
        np.sum(pvals < 0.05),
        np.sum(reject_bonf),
        np.sum(reject_bh),
        np.sum(reject_qval)
    ],
    'Notes': [
        'No correction — massively inflated',
        'Controls FWER — very conservative',
        'Controls FDR — assumes pi0 = 1',
        f'Controls FDR — uses pi0 = {pi0_hat:.3f}'
    ]
})

print(summary.to_string(index=False))

# Visual comparison
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
bars = ax.barh(summary['Method'], summary['Discoveries'], color=colors, edgecolor='white')
ax.set_xlabel('Number of Discoveries')
ax.set_title('Method Comparison: Discoveries at 5% Threshold')
for bar, val in zip(bars, summary['Discoveries']):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2, 
            str(val), va='center', fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 7: AI Extension — Local FDR

Efron's **local FDR** gives a gene-level probability of being null:

$$\text{fdr}(z) = \frac{\pi_0 f_0(z)}{f(z)}$$

where $f_0(z)$ is the null density and $f(z)$ is the overall mixture density. This is more informative than the "tail-area" FDR from BH, because it tells you about each individual gene rather than a set of genes.

**Your task**: Ask an AI to implement local FDR in Python.

**Suggested prompt**: "Implement Efron's local FDR in Python. Given z-scores from gene-level t-tests, estimate the mixture density f(z) using kernel density estimation, estimate the empirical null f0(z) by fitting a normal to central z-scores, estimate pi0, and compute fdr(z) = pi0 * f0(z) / f(z) for each gene. Plot the local FDR as a function of z-score."

Paste the AI-generated code in the cell below and run it.

In [ ]:
# YOUR CODE HERE (AI-generated or your own)


### Critique Checklist

| Question | Your Answer |
|----------|-------------|
| Did it estimate $f(z)$ nonparametrically (e.g., KDE)? | |
| Did it fit the empirical null from central z-scores only (e.g., $|z| < 2$)? | |
| Is the local fdr between 0 and 1 for all genes? | |
| Does the local fdr curve make sense (high in the middle, low in the tails)? | |
| Did it handle edge cases (e.g., fdr > 1 clipped to 1)? | |

---

## Wrap-Up Questions

1. Why does the BH procedure give different results from q-values? Which is more powerful?

**YOUR ANSWER:**

2. If the empirical null has $\sigma_0 > 1$, and you used $N(0,1)$ instead, would you get more or fewer discoveries? Would they be reliable?

**YOUR ANSWER:**

3. When would you report BH-adjusted p-values vs. q-values in a paper?

**YOUR ANSWER:**

4. The p-value histogram is the key diagnostic. Name three things it tells you.

**YOUR ANSWER:**

---

## Looking Ahead

You now have a list of differentially expressed genes with FDR control. But you've only answered *which genes differ*. In **Lab 3**, you'll ask whether the samples have lower-dimensional structure (PCA, UMAP), and in **Lab 4**, you'll build a predictive gene signature using penalized regression.